# Movies dataset

This notebook loads a small movie-domain graph into both `NetworkXGraph` and
`Neo4jGraph`.

Dataset notes:
- movie nodes with title and IMDb id
- genre nodes linked with `IN_GENRE`
- source is a public sample API, with a bundled fallback for offline use

In [ ]:
import importlib.util

# Comprovació de presència del paquet
package_to_check = 'cvcdocdb'
spec = importlib.util.find_spec(package_to_check)

if spec is None:
    print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
    %pip install -q --upgrade cvcdocdb-tools
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')


In [ ]:
import os

from cvcdocdb import Neo4jGraph, NetworkXGraph
from cvcdocdb.exemples import load_movies_sample


def neo4j_config(default_target="DEV"):
    target = os.environ.get("NEO4J_TARGET", default_target).upper()
    prefix = f"NEO4J_{target}_"
    return {
        "target": target,
        "url": os.environ.get(f"{prefix}URL", os.environ.get("NEO4J_URL", "bolt://localhost:7687")),
        "user": os.environ.get(f"{prefix}USER", os.environ.get("NEO4J_USER", "neo4j")),
        "password": os.environ.get(f"{prefix}PASSWORD", os.environ.get("NEO4J_PASSWORD", "secret")),
        "database": os.environ.get(f"{prefix}DATABASE", os.environ.get("NEO4J_DATABASE", "neo4j")),
    }

## NetworkXGraph

In [ ]:
nx_graph = NetworkXGraph()
# load_movies_sample works with both NetworkXGraph (in-memory) and Neo4jGraph (persistent)
nx_stats = load_movies_sample(nx_graph, limit=25)
print("NetworkX stats:", nx_stats)
nx_graph.close()

## Neo4jGraph

In [ ]:
cfg = neo4j_config()
print("Using Neo4j target:", cfg["target"])
neo4j_graph = Neo4jGraph(
    url=cfg["url"],
    user=cfg["user"],
    password=cfg["password"],
    database=cfg["database"],
)
try:
    neo4j_stats = load_movies_sample(neo4j_graph, limit=25)
    print("Neo4j stats:", neo4j_stats)
finally:
    neo4j_graph.close()
